# Ablation: without latitude, longitude and tilt (Tables 9–10)

## Setup

In [2]:
import sys, os, dataclasses
sys.path.insert(0, "../..")

from solar_ml_core import *

import numpy as np
import pandas as pd
from scipy import stats

DROP = ("Latitude", "Longitude", "Tilt")
CFG_AB = dataclasses.replace(
    CFG,
    base_features=tuple(f for f in CFG.base_features if f not in DROP),
    results_folder_name="ablation",
)
for d in (CFG_AB.results_dir, CFG_AB.predictions_dir, CFG_AB.models_dir):
    os.makedirs(d, exist_ok=True)

ALGOS_AB = ["MLR", "KNN", "SVR", "RandomForest", "XGBoost", "LSTM"]
SITES = ["Atacama", "Bolzano", "Elazig", "Helsinki", "Islamabad", "Ningxia", "Yulara"]

## Run LOSO without station-constant features

In [3]:
def is_done(a):
    res = os.path.join(CFG_AB.results_dir, f"results_{a}.csv")
    preds = [os.path.join(CFG_AB.predictions_dir, f"{a}_{s}.csv") for s in SITES]
    return os.path.exists(res) and all(os.path.exists(p) for p in preds)

FOLDS = load_loso_folds(CFG_AB)
for a in ALGOS_AB:
    if not is_done(a):
        run_algorithm(a, FOLDS, CFG_AB)

Loading LOSO folds
[Atacama   ] train 52,584 rows (6 stations) | test 8,676 rows | flagged %10.0
[Bolzano   ] train 52,500 rows (6 stations) | test 8,760 rows | flagged %10.1
[Elazig    ] train 52,500 rows (6 stations) | test 8,760 rows | flagged %10.1
[Helsinki  ] train 52,500 rows (6 stations) | test 8,760 rows | flagged %8.5
[Islamabad ] train 52,500 rows (6 stations) | test 8,760 rows | flagged %10.1
[Ningxia   ] train 52,500 rows (6 stations) | test 8,760 rows | flagged %10.0
[Yulara    ] train 52,476 rows (6 stations) | test 8,784 rows | flagged %10.1
7 folds.


## Table 9: all features vs. ablation

In [4]:
ORIG, ABL = "All features", "Without Lat/Lon/Tilt"

def metrics(y, yh):
    e = yh - y
    return {"R2": 1 - (e @ e) / ((y - y.mean()) ** 2).sum(),
            "RMSE": float(np.sqrt((e @ e) / len(y))), "MAE": float(np.abs(e).mean())}

def load(cfg, algos, tag):
    rows = []
    for a in algos:
        for s in SITES:
            p = pd.read_csv(os.path.join(cfg.predictions_dir, f"{a}_{s}.csv"))
            if "used_in_metrics" in p.columns:
                p = p[p["used_in_metrics"]]
            y = p["y_true"].to_numpy(float)
            yh = np.clip(p["y_pred"].to_numpy(float), 0.0, None)
            rows.append({"Model": a, "Test_Station": s, "Setup": tag, **metrics(y, yh)})
    return pd.DataFrame(rows)

ALL = pd.concat([load(CFG, ALGOS_AB, ORIG), load(CFG_AB, ALGOS_AB, ABL)], ignore_index=True)

M = ALL.groupby(["Model", "Setup"])[["R2", "RMSE", "MAE"]].mean().unstack("Setup")
for m in ("R2", "RMSE", "MAE"):
    M[(m, "Delta")] = M[(m, ABL)] - M[(m, ORIG)]
display(M.sort_index(axis=1).loc[ALGOS_AB].round(4))

P = ALL.pivot_table(index="Test_Station", columns=["Model", "Setup"], values="MAE")
D = pd.DataFrame({a: P[(a, ABL)] - P[(a, ORIG)] for a in ALGOS_AB})
display(D.loc[SITES].round(4))
print("Stations where MAE decreased:", (D < 0).sum().to_dict())

MAE                                        R2          \
Setup        All features   Delta Without Lat/Lon/Tilt All features   Delta   
Model                                                                         
MLR                0.0637 -0.0116               0.0521       0.8307 -0.0462   
KNN                0.0571  0.0002               0.0573       0.7222  0.0223   
SVR                0.0887 -0.0396               0.0491       0.7304  0.0488   
RandomForest       0.0547 -0.0015               0.0532       0.7694  0.0038   
XGBoost            0.0520  0.0006               0.0526       0.7939 -0.0270   
LSTM               0.0748 -0.0077               0.0671       0.7020 -0.0032   

                                          RMSE                               
Setup        Without Lat/Lon/Tilt All features   Delta Without Lat/Lon/Tilt  
Model                                                                        
MLR                        0.7845       0.0963 -0.0008               0.0955  
KNN                        0.7445       0.1023  0.0059               0.1082  
SVR                        0.7791       0.1232 -0.0253               0.0978  
RandomForest               0.7733       0.1092 -0.0035               0.1058  
XGBoost                    0.7668       0.1029  0.0019               0.1047  
LSTM                       0.6988       0.1288 -0.0037               0.1251

,MLR,KNN,SVR,RandomForest,XGBoost,LSTM
Test_Station,,,,,,
Atacama,-0.0750,0.0372,-0.2015,0.0058,-0.0112,-0.0698
Bolzano,-0.0061,0.0008,-0.0089,-0.0072,-0.0003,0.0045
Elazig,-0.0011,0.0006,-0.0098,0.0016,0.0028,0.0171
Helsinki,-0.0163,-0.0043,-0.0425,-0.0071,-0.0082,-0.0225
Islamabad,0.0081,-0.0065,0.0036,0.0053,0.0023,0.0097
Ningxia,-0.0172,-0.0023,-0.0122,-0.0156,0.0052,0.0043
Yulara,0.0260,-0.0241,-0.0059,0.0069,0.0136,0.0030


Stations where MAE decreased: {'MLR': 5, 'KNN': 4, 'SVR': 6, 'RandomForest': 3, 'XGBoost': 3, 'LSTM': 2}


## Physical reference (k·POA)

In [5]:
ref = {}
for s, (tr, te) in FOLDS.items():
    tr = tr.dropna(subset=[CFG.poa_feature, CFG.target_col])
    te = te.dropna(subset=[CFG.poa_feature, CFG.target_col])
    x, y = tr[CFG.poa_feature].to_numpy(float), tr[CFG.target_col].to_numpy(float)
    k = (x @ y) / (x @ x)
    yt = te[CFG.target_col].to_numpy(float)
    ref[s] = metrics(yt, np.clip(k * te[CFG.poa_feature].to_numpy(float), 0, None))
REF = pd.DataFrame(ref).T

## Table 10: skill score, Friedman (Iman–Davenport) and Holm

In [6]:
CONTROL = "Reference"

R2_AB = ALL[ALL.Setup == ABL].pivot(index="Test_Station", columns="Model", values="R2")
R2_AB.insert(0, CONTROL, REF["R2"])
display(R2_AB.loc[SITES, [CONTROL] + ALGOS_AB].round(3))

def skill(setup):
    r = ALL[ALL.Setup == setup].pivot(index="Test_Station", columns="Model", values="RMSE")
    return r.loc[SITES, ALGOS_AB].apply(lambda c: 1 - c / REF.loc[SITES, "RMSE"])

for setup in (ORIG, ABL):
    S = skill(setup)
    display(pd.concat([S, S.median().to_frame("Median").T,
                       (S > 0).sum().to_frame("Stations beating reference").T]).round(3))

def friedman_holm(M, higher_is_better, alpha=0.05):
    N, k = M.shape
    ranks = M.rank(axis=1, ascending=not higher_is_better).mean()
    chi2 = stats.friedmanchisquare(*[M[c] for c in M.columns]).statistic
    FF = (N - 1) * chi2 / (N * (k - 1) - chi2)
    p = stats.f.sf(FF, k - 1, (k - 1) * (N - 1))
    print(f"N={N}, k={k} | F={FF:.2f}, p={p:.4f}")
    print("mean ranks:", ranks.sort_values().round(2).to_dict())
    if p >= alpha:
        return
    SE = np.sqrt(k * (k + 1) / (6 * N))
    df = pd.DataFrame([{"Model": m, "Rank_diff": ranks[m] - ranks[CONTROL]} for m in M.columns if m != CONTROL])
    df["p"] = 2 * stats.norm.sf((df.Rank_diff / SE).abs())
    df = df.sort_values("p").reset_index(drop=True)
    df["Holm_threshold"] = [alpha / (k - 1 - i) for i in range(len(df))]
    sig, stop = [], False
    for p_, a_ in zip(df.p, df.Holm_threshold):
        s_ = (not stop) and p_ < a_; stop = stop or not s_; sig.append(s_)
    df["Significant"] = sig
    df["Direction"] = np.where(df.Rank_diff > 0, "Reference better", "Model better")
    display(df.round(4))

for metric, hib in (("R2", True), ("MAE", False)):
    M = ALL[ALL.Setup == ABL].pivot(index="Test_Station", columns="Model", values=metric)
    M.insert(0, CONTROL, REF[metric])
    print(f"\nAblation, {metric}")
    friedman_holm(M.loc[SITES, [CONTROL] + ALGOS_AB], hib)

Model,Reference,MLR,KNN,SVR,RandomForest,XGBoost,LSTM
Test_Station,,,,,,,
Atacama,0.934,0.939,0.832,0.881,0.781,0.873,0.905
Bolzano,0.884,0.826,0.861,0.900,0.887,0.879,0.764
Elazig,0.790,0.812,0.807,0.799,0.806,0.810,0.748
Helsinki,0.925,0.915,0.897,0.925,0.884,0.848,0.737
Islamabad,0.986,0.952,0.908,0.925,0.925,0.921,0.807
Ningxia,0.925,0.960,0.941,0.962,0.943,0.927,0.802
Yulara,0.081,0.088,-0.034,0.061,0.186,0.111,0.128


Model,MLR,KNN,SVR,RandomForest,XGBoost,LSTM
Atacama,-0.649,0.306,-2.369,-0.543,-0.643,-1.319
Bolzano,-0.096,-0.068,0.079,-0.058,-0.031,-0.256
Elazig,0.094,0.006,-0.078,0.007,0.037,0.106
Helsinki,-0.234,-0.175,-0.772,-0.348,-0.436,-1.773
Islamabad,-0.547,-1.869,-0.698,-1.218,-1.163,-2.039
Ningxia,0.056,0.064,-0.088,-0.511,0.156,-0.286
Yulara,0.279,-0.183,0.317,0.102,0.145,0.228
Median,-0.096,-0.068,-0.088,-0.348,-0.031,-0.286
Stations beating reference,3.000,3.000,2.000,2.000,3.000,2.000


Model,MLR,KNN,SVR,RandomForest,XGBoost,LSTM
Atacama,0.039,-0.596,-0.341,-0.820,-0.386,-0.200
Bolzano,-0.227,-0.097,0.072,0.013,-0.025,-0.429
Elazig,0.052,0.039,0.022,0.038,0.046,-0.098
Helsinki,-0.059,-0.166,-0.000,-0.239,-0.423,-0.870
Islamabad,-0.840,-1.544,-1.295,-1.295,-1.368,-2.695
Ningxia,0.272,0.118,0.291,0.129,0.016,-0.622
Yulara,0.004,-0.061,-0.011,0.059,0.017,0.027
Median,0.004,-0.097,-0.000,0.013,-0.025,-0.429
Stations beating reference,4.000,2.000,3.000,4.000,3.000,1.000



Ablation, R2
N=7, k=7 | F=1.93, p=0.1023
mean ranks: {'MLR': 2.71, 'SVR': 3.14, 'Reference': 3.43, 'RandomForest': 3.71, 'XGBoost': 4.29, 'KNN': 5.0, 'LSTM': 5.71}

Ablation, MAE
N=7, k=7 | F=4.89, p=0.0010
mean ranks: {'SVR': 2.29, 'Reference': 2.57, 'RandomForest': 3.43, 'MLR': 4.0, 'XGBoost': 4.0, 'KNN': 5.43, 'LSTM': 6.29}


,Model,Rank_diff,p,Holm_threshold,Significant,Direction
0,LSTM,3.7143,0.0013,0.0083,True,Reference better
1,KNN,2.8571,0.0133,0.0100,False,Reference better
2,XGBoost,1.4286,0.2160,0.0125,False,Reference better
3,MLR,1.4286,0.2160,0.0167,False,Reference better
4,RandomForest,0.8571,0.4579,0.0250,False,Reference better
5,SVR,-0.2857,0.8046,0.0500,False,Model better
